In [1]:
from google.colab import files

uploaded = files.upload()

Saving FYP_synthetic_vehicle_fault_dataset_50000_checked.csv to FYP_synthetic_vehicle_fault_dataset_50000_checked.csv


In [2]:

import pandas as pd
from google.colab import files
from sklearn.model_selection import GroupShuffleSplit

uploaded = files.upload()
file_name = next(iter(uploaded))
df = pd.read_csv(file_name)

print("Dataset shape:", df.shape)
print("\nColumns:", df.columns.tolist())

features = [
    "engine_coolant_temperature",
    "oil_pressure_kpa",
    "engine_rpm",
    "vehicle_speed",
    "throttle_position",
    "engine_load"
]
target = "condition"
group_column = "sequence_id"

required = features + [target, group_column]
missing_columns = [c for c in required if c not in df.columns]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

print("\nMissing values:")
print(df[required].isnull().sum())

print("\nCondition label counts:")
print(df[target].value_counts(dropna=False))

print("\nUnique sequences:", df[group_column].nunique())

if df[required].isnull().any().any():
    raise ValueError("Missing values found. Investigate before splitting.")

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(df[features], df[target], groups=df[group_column])
)

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

X_train = train_df[features]
y_train = train_df[target]

X_test = test_df[features]
y_test = test_df[target]

train_sequences = set(train_df[group_column])
test_sequences = set(test_df[group_column])

assert train_sequences.isdisjoint(test_sequences)

print("\nTraining rows:", len(train_df))
print("Testing rows:", len(test_df))
print("Training sequences:", len(train_sequences))
print("Testing sequences:", len(test_sequences))
print("Sequence overlap:", len(train_sequences & test_sequences))

print("\nTraining label distribution:")
print(y_train.value_counts())

print("\nTesting label distribution:")
print(y_test.value_counts())

train_df.to_csv("train_condition.csv", index=False)
test_df.to_csv("test_condition.csv", index=False)

print("\nTrain/test datasets saved successfully.")


Saving FYP_synthetic_vehicle_fault_dataset_50000_checked.csv to FYP_synthetic_vehicle_fault_dataset_50000_checked (1).csv
Dataset shape: (50000, 12)

Columns: ['sequence_id', 'timestamp_step', 'engine_coolant_temperature', 'oil_pressure_kpa', 'engine_rpm', 'vehicle_speed', 'throttle_position', 'engine_load', 'scenario_type', 'fault_type', 'condition', 'fault_active']

Missing values:
engine_coolant_temperature    0
oil_pressure_kpa              0
engine_rpm                    0
vehicle_speed                 0
throttle_position             0
engine_load                   0
condition                     0
sequence_id                   0
dtype: int64

Condition label counts:
condition
normal      38299
critical     6975
warning      4726
Name: count, dtype: int64

Unique sequences: 500

Training rows: 40000
Testing rows: 10000
Training sequences: 400
Testing sequences: 100
Sequence overlap: 0

Training label distribution:
condition
normal      30782
critical     5378
warning      3840
Nam

In [3]:

files.download("train_condition.csv")
files.download("test_condition.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>